![DSPy](/workspace/src/assets/dspy_logo.png)

DSPy (Declarative Self-improving Python) es un framework para programar aplicaciones basadas en modelos de lenguaje. En lugar de centrar el desarrollo en escribir y ajustar prompts manualmente, DSPy permite definir qué debe hacer el sistema, cómo medir si lo hace correctamente y utilizar esa información para optimizar el programa automáticamente.

![alt text](assets/promt_vs_dspy.png)

## Flujos de trabajo

### Prompt

![alt text](assets/w_prompt.png)

### DSPY

![alt text](assets/w_dspy.png)

![alt text](assets/w_dspy_2.png)

____

## Set up

In [1]:
# import dspy

# lm = dspy.LM("modelo_que_queremos_usar", api_key="YOUR_OPENAI_API_KEY")

# dspy.configure(lm=lm)

In [24]:
import os
import importlib
import lm_config as dspy_lm_config

from lm_config import configure_lm
import dspy


os.environ["DSPY_LM_BACKEND"] = "ollama"
os.environ["OLLAMA_BASE_URL"] = "http://ollama:11434"
os.environ["OLLAMA_MODEL"] = "llama3.2:latest"
# os.environ["OLLAMA_MODEL"] = "phi3:latest"

importlib.reload(dspy_lm_config)


lm = configure_lm(max_tokens=1024)
print(lm.model)

ollama_chat/llama3.2:latest


_____

## 01 - Signatures - Que quiero hacer

Una <strong style="color:#c2410c">Signature</strong> se puede entender como una tarea que quiero realizar describiendo la a travez de sus entradas y salidas.
La idea es definir una tarea simple:
- input → pregunta
- output → respuesta

DSPy se encargará de construir internamente el prompting necesario.

![alt text](assets/signature.png)

In [25]:
texto = "ingles -> espanol"
traducir = dspy.Predict(texto)
resultado = traducir(ingles="The security deposit equals two months of rent.")
print(resultado.espanol)

El depósito de seguridad equivale a dos meses de alquiler.


In [26]:
reescribir = dspy.Predict("texto: str, tono: str -> reescritura: str")
r = reescribir(
    texto="hay un recargo en caso de no pagar a tiempo",
    tono="formal y educado",
)
print(r.reescritura)

se aplicará un recargo en el caso de no pagar a tiempo


## 02 - Class-based Signatures

Las String Signatures son muy cómodas para comenzar, pero rápidamente se quedan cortas cuando queremos agregar instrucciones, descripciones o tipos más complejos. Para resolver esto, DSPy ofrece las <strong style="color:#c2410c">Class-based Signatures</strong>.

In [27]:
class DocumentSummary(dspy.Signature):
    """Resume un documento para una audiencia determinada."""

    documento: str = dspy.InputField()
    audiencia: str = dspy.InputField()
    max_palabras: int = dspy.InputField()

    resumen: str = dspy.OutputField()
    titulo: str = dspy.OutputField()
    palabras_clave: list[str] = dspy.OutputField()

In [28]:
summarizer = dspy.Predict(DocumentSummary)

In [29]:
resultado = summarizer(
    documento="""
    DSPy es un framework para construir aplicaciones con modelos de lenguaje
    utilizando Signatures y Modules en lugar de prompts escritos manualmente.
    """,
    audiencia="Desarrolladores Python principiantes",
    max_palabras=50
)

In [30]:
print(resultado.titulo)
print(resultado.resumen)
print(resultado.palabras_clave)

Introducción a DSPy
DSPy es un framework que permite construir aplicaciones con modelos de lenguaje utilizando Signatures y Modules, lo que facilita la creación de aplicaciones sin necesidad de escribir prompts manualmente.
['DSPy', 'framework', 'modelos de lenguaje', 'Signatures', 'Modules']


![alt text](assets/class-base-signatu.png)

In [31]:
from typing import Literal

Audiencia = Literal[
    "principiantes",
    "tecnicos",
    "ejecutivos",
]

class DocumentSummary(dspy.Signature):
    """
    Resume un documento según la audiencia.
    """
    documento: str = dspy.InputField()
    audiencia: Audiencia = dspy.InputField()
    resumen: str = dspy.OutputField()
    titulo: str = dspy.OutputField()

In [32]:
summarizer = dspy.Predict(DocumentSummary)

In [35]:

resultado = summarizer(
    documento="""
    DSPy permite construir aplicaciones
    utilizando Signatures y Modules.
    """,
    audiencia="principiantes",
)

print(resultado.titulo)
print(resultado.resumen)

Introducción a DSPy
DSPy es una plataforma para desarrollar aplicaciones utilizando Signatures y Modules, lo que permite una mayor flexibilidad y personalización en el proceso de creación.


In [34]:

resultado = summarizer(
    documento="""
    DSPy permite construir aplicaciones
    utilizando Signatures y Modules.
    """,
    audiencia="ejecutivos",
)

print(resultado.titulo)
print(resultado.resumen)

Construyendo Aplicaciones con DSPy a Través de Signatures y Modules
El documento DSPy proporciona una forma flexible de construir aplicaciones mediante el uso de Signatures y Modules.


### 02.1- Datos estructurados

Muchas aplicaciones necesitan devolver información más compleja. En lugar de devolver varios campos independientes, podemos agruparlos en un único objeto utilizando <strong style="color:#c2410c">Pydantic</strong>. Esto hace que nuestras `Signatures` sean más expresivas y que el resultado sea mucho más fácil de utilizar desde Python.

In [36]:
from pydantic import BaseModel

class ResumenDocumento(BaseModel):
    titulo: str
    resumen: str
    palabras_clave: list[str]


class DocumentSummary(dspy.Signature):
    """Resume un documento según la audiencia."""
    documento: str = dspy.InputField()
    audiencia: Audiencia = dspy.InputField()
    resultado: ResumenDocumento = dspy.OutputField()

In [37]:
summarizer = dspy.Predict(DocumentSummary)

In [46]:
resultado = summarizer(
    documento="""
    DSPy permite construir aplicaciones
    utilizando Signatures y Modules.
    """,
    audiencia="principiantes",
)

print(resultado.resultado.titulo)
print(resultado.resultado.resumen)
print(resultado.resultado.palabras_clave)


Resumen del documento
El documento describe cómo se pueden construir aplicaciones utilizando DSPy, una herramienta de desarrollo que permite la creación de aplicaciones utilizando firmas y módulos.
['DSPy', 'aplicaciones', 'Signatures', 'Modules']


In [47]:
resultado = summarizer(
    documento="""
    DSPy permite construir aplicaciones
    utilizando Signatures y Modules.
    """,
    audiencia="ejecutivos",
)

print(resultado.resultado.titulo)
print(resultado.resultado.resumen)
print(resultado.resultado.palabras_clave)


Características de DSPy
DSPy es una herramienta para construir aplicaciones utilizando Signatures y Modules.
['aplicaciones', 'Signatures', 'Modules']


## 03 - Modules - Como lo quiero hacer

Los <strong style="color:#c2410c">Modules</strong> implementan distintas estrategias para ejecutar una Signature. Podemos mantener la misma tarea y cambiar el Module para modificar la forma en que el modelo intenta resolverla.

![alt text](assets/Modules.png)